L'objectif de ce notebook est de préparer les données transactionnelles afin de construire un dataset analytique permettant d'étudier les performances quotidiennes des trois points de vente.

Pour cela, nous allons créer les variables et indicateurs nécessaires à l'analyse, notamment le chiffre d'affaires net quotidien, le nombre de transactions et le panier moyen.

Ce dataset servira de base à notre analyse des variations de performance entre les journées et les établissements.

In [1]:
import pandas as pd
import numpy as np

transactions = pd.read_csv('../data/raw/Transactions.csv')
transactions.head() 

,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0


In [3]:
transactions.dtypes


Outlet                str
Date                  str
Time                  str
NetSales          float64
Tax               float64
TotalAmount         int64
ReceiptNumber         str
Items                 str
TotalItem           int64
PaymentMethod         str
UseLoyaltyCard      int64
dtype: object

In [4]:
# conversion des variables Date et Time

transactions['Date'] = pd.to_datetime(transactions['Date'], format='%Y-%m-%d')
transactions['Time'] = pd.to_datetime(transactions['Time'], format='%H:%M:%S').dt.time

In [5]:
transactions[['Date', 'Time']].dtypes

Date    datetime64[us]
Time            object
dtype: object

In [6]:
transactions[['Date', 'Time']].head()

,Date,Time
0,2025-01-02,07:01:05
1,2025-01-02,07:04:11
2,2025-01-02,07:10:46
3,2025-01-02,07:14:25
4,2025-01-02,07:23:05


Construction du dataset analytique

In [9]:
# création de la performance quotidienne avec les variables de transactions

performance_quotidienne = transactions.groupby(['Date', 'Outlet']).agg(
    CA_net=('NetSales', 'sum'), Nb_transactions=('ReceiptNumber', 'nunique'), Nb_articles=('TotalItem', 'sum')
    ).reset_index()

In [10]:
performance_quotidienne.head(10)

,Date,Outlet,CA_net,Nb_transactions,Nb_articles
0,2025-01-02,SHOP001,1808108.121,82,94
1,2025-01-02,SHOP002,1116216.221,51,64
2,2025-01-02,SHOP003,1234234.242,58,66
3,2025-01-03,SHOP001,2334234.248,101,122
4,2025-01-03,SHOP002,486486.490,26,26
5,2025-01-03,SHOP003,1695495.507,76,89
6,2025-01-04,SHOP001,2227927.944,99,115
7,2025-01-04,SHOP002,609909.914,28,33
8,2025-01-04,SHOP003,972972.978,50,55
9,2025-01-05,SHOP001,1904504.519,90,106


In [11]:
#Calcul du panier moyen 

performance_quotidienne['Panier_moyen'] = performance_quotidienne['CA_net'] / performance_quotidienne['Nb_transactions']

In [12]:
performance_quotidienne.head(10)

,Date,Outlet,CA_net,Nb_transactions,Nb_articles,Panier_moyen
0,2025-01-02,SHOP001,1808108.121,82,94,22050.099037
1,2025-01-02,SHOP002,1116216.221,51,64,21886.592569
2,2025-01-02,SHOP003,1234234.242,58,66,21279.900724
3,2025-01-03,SHOP001,2334234.248,101,122,23111.230178
4,2025-01-03,SHOP002,486486.490,26,26,18711.018846
5,2025-01-03,SHOP003,1695495.507,76,89,22309.151408
6,2025-01-04,SHOP001,2227927.944,99,115,22504.322667
7,2025-01-04,SHOP002,609909.914,28,33,21782.496929
8,2025-01-04,SHOP003,972972.978,50,55,19459.459560
9,2025-01-05,SHOP001,1904504.519,90,106,21161.161322


In [13]:
performance_quotidienne['Jour_semaine'] = performance_quotidienne['Date'].dt.day_name()
performance_quotidienne['Mois'] = performance_quotidienne['Date'].dt.month

In [14]:
performance_quotidienne.head(10)

,Date,Outlet,CA_net,Nb_transactions,Nb_articles,Panier_moyen,Jour_semaine,Mois
0,2025-01-02,SHOP001,1808108.121,82,94,22050.099037,Thursday,1
1,2025-01-02,SHOP002,1116216.221,51,64,21886.592569,Thursday,1
2,2025-01-02,SHOP003,1234234.242,58,66,21279.900724,Thursday,1
3,2025-01-03,SHOP001,2334234.248,101,122,23111.230178,Friday,1
4,2025-01-03,SHOP002,486486.490,26,26,18711.018846,Friday,1
5,2025-01-03,SHOP003,1695495.507,76,89,22309.151408,Friday,1
6,2025-01-04,SHOP001,2227927.944,99,115,22504.322667,Saturday,1
7,2025-01-04,SHOP002,609909.914,28,33,21782.496929,Saturday,1
8,2025-01-04,SHOP003,972972.978,50,55,19459.459560,Saturday,1
9,2025-01-05,SHOP001,1904504.519,90,106,21161.161322,Sunday,1


In [15]:
print('Doublons Date-Outlet :', performance_quotidienne.duplicated(subset=['Date', 'Outlet']).sum())
print('Valeurs manquantes :', performance_quotidienne.isna().sum())
print('Transactions nulles :', (performance_quotidienne['Nb_transactions'] == 0).sum())

Doublons Date-Outlet : 0
Valeurs manquantes : Date               0
Outlet             0
CA_net             0
Nb_transactions    0
Nb_articles        0
Panier_moyen       0
Jour_semaine       0
Mois               0
dtype: int64
Transactions nulles : 0


In [16]:
print('CA source :', transactions['NetSales'].sum())
print('CA agrégé :', performance_quotidienne['CA_net'].sum())
print('Transactions source :', transactions['ReceiptNumber'].nunique())
print('Transactions agrégées :', performance_quotidienne['Nb_transactions'].sum())
print('Articles source :', transactions['TotalItem'].sum())
print('Articles agrégés :', performance_quotidienne['Nb_articles'].sum())

CA source : 1184933340.536
CA agrégé : 1184933340.5359998
Transactions source : 53820
Transactions agrégées : 53820
Articles source : 61973
Articles agrégés : 61973


In [17]:
performance_quotidienne.to_csv('../data/processed/performance_quotidienne.csv', index=False)